In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [9]:
# 1. Cargar los datos
df = pd.read_csv('doctors_aggregated.csv')

# 2. Limpieza inicial: Separar etiquetados de no etiquetados
# Nota: Ajusta '0' si el valor de "no clasificado" es distinto (ej. NaN o "SIN SEGMENTO")
df_labeled = df[df['ATSEG_first'] != '0'].copy()
df_unlabeled = df[df['ATSEG_first'] == '0'].copy()

# 3. Definir X e y (eliminando IDs y columnas de fechas)
# Eliminamos columnas que no son predictoras numéricas
cols_to_drop = ['NUEVO_ID', 'ATSEG_first', 'WEEK_ID_first', 'WEEK_ID_last']
X = df_labeled.drop(columns=cols_to_drop)
y = df_labeled['ATSEG_first']

# 4. División Train/Test (¡IMPORTANTE: Antes de escalar y balancear!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Escalado de datos
scaler = StandardScaler()

# Ajustamos el escalador solo con los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)
# Transformamos el test con la media y desviación del train
X_test_scaled = scaler.transform(X_test)

# 6. Aplicar SMOTE para balancear los segmentos A, B y C
# Solo lo aplicamos al set de ENTRENAMIENTO
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# --- Verificación de resultados ---
print(f"--- Resumen de Datos ---")
print(f"Total registros etiquetados: {len(df_labeled)}")
print(f"Total registros por clasificar: {len(df_unlabeled)}")
print("\nDistribución original en Train:")
print(y_train.value_counts())
print("\nDistribución después de SMOTE en Train:")
print(pd.Series(y_train_resampled).value_counts())

# --- BONUS: Preparar los 'No Clasificados' para predecir después ---
if not df_unlabeled.empty:
    X_unlabeled = df_unlabeled.drop(columns=cols_to_drop)
    # Usamos el mismo escalador que entrenamos arriba
    X_unlabeled_scaled = scaler.transform(X_unlabeled)
    print(f"\nDatos listos para predicción: {X_unlabeled_scaled.shape}")

--- Resumen de Datos ---
Total registros etiquetados: 11899
Total registros por clasificar: 9032

Distribución original en Train:
ATSEG_first
SEG_A    5125
SEG_B    2679
SEG_C    1715
Name: count, dtype: int64

Distribución después de SMOTE en Train:
ATSEG_first
SEG_A    5125
SEG_C    5125
SEG_B    5125
Name: count, dtype: int64

Datos listos para predicción: (9032, 147)


In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import LabelEncoder

# 1. Convertir etiquetas de texto (A, B, C) a números (0, 1, 2)
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train_resampled)
y_test_encoded = encoder.transform(y_test)

# 2. Definir la arquitectura
model = models.Sequential([
    # Primera capa: input_shape debe ser el número de columnas de tus datos
    layers.Dense(64, activation='relu', input_shape=(X_train_resampled.shape[1],)),
    layers.Dropout(0.2), # Apaga el 20% de neuronas al azar para evitar overfitting
    
    # Segunda capa oculta
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    
    # Capa de salida: 3 clases (A, B, C)
    layers.Dense(3, activation='softmax') 
])

# 3. Compilar el modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Usamos esta porque y son números (0,1,2)
    metrics=['accuracy']
)

model.summary()

ModuleNotFoundError: No module named 'tensorflow.python'

In [18]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl (351.2 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\azihu\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\azihu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
